# CIPHER v2 — RED & BLUE Commander Training (Kaggle T4)

Trains the **two top-level commanders** for the new commander + dynamic-subagent architecture (`CIPHER_AGENT_ARCH=v2`).

- Base model: `unsloth/Llama-3.2-1B-Instruct` (4-bit)
- LoRA: r=16, alpha=32 (q,k,v,o + gate/up/down)
- Dataset: 200 real CIPHER stub-mode episodes per commander, observations + commander-context blocks pulled directly from the live registry
- Reward: validates JSON shape, primitive vs meta-action choice, role appropriateness, spawn-budget discipline, and zone-aware actions
- Training: 8 epochs each (~25 min each on T4) — total ≈ 60 min

**Outputs** (zipped at the end so you can download from the Kaggle session):
- `/kaggle/working/cipher-red-commander-v1/` → drop into project root as `red trained/cipher-red-commander-v1/`
- `/kaggle/working/cipher-blue-commander-v1/` → drop into project root as `blue trained/cipher-blue-commander-v1/`

Once placed, `python main.py --hybrid` will pick them up automatically via `RED_COMMANDER_LORA_PATH` / `BLUE_COMMANDER_LORA_PATH` in `.env`.

**Before running:** Settings → Internet → ON | Accelerator → GPU T4

In [ ]:
# Cell 1 — Verify GPU
import subprocess, os
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'NO GPU — enable in Kaggle Settings -> Accelerator')

KAGGLE_OUT = '/kaggle/working'
os.makedirs(KAGGLE_OUT, exist_ok=True)
print(f'Output path: {KAGGLE_OUT}')

In [ ]:
# Cell 2 — Install dependencies + clone CIPHER repo
import subprocess, sys, os

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'WARN: {r.stderr[:300]}')

run('pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q')
run('pip install "trl>=0.12.0" "datasets>=2.19.0" "accelerate>=0.30.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" -q')

# Replace the URL below with your own clone target (or upload as a Kaggle dataset).
CIPHER_PATH = '/kaggle/working/CIPHER/cipher'
if not os.path.exists(CIPHER_PATH):
    run(f'git clone https://github.com/Rishaan08/CIPHER {CIPHER_PATH}')

if CIPHER_PATH not in sys.path:
    sys.path.insert(0, CIPHER_PATH)
run(f'pip install -e {CIPHER_PATH} -q')

print('All dependencies installed')
print(f'CIPHER path: {CIPHER_PATH}')

In [ ]:
# Cell 3 — Verify setup
import sys, os, torch

CIPHER_PATH = '/kaggle/working/CIPHER/cipher'
if CIPHER_PATH not in sys.path:
    sys.path.insert(0, CIPHER_PATH)
os.environ['LLM_MODE'] = 'stub'
os.environ['CIPHER_AGENT_ARCH'] = 'v2'

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from cipher.utils.config import config
from cipher.agents.commander import RedCommander, BlueCommander
from cipher.agents.role_profiles import list_profiles

print(f'CIPHER arch: {config.cipher_agent_arch}')
print(f'RED roles  : {[p.role_name for p in list_profiles("red")]}')
print(f'BLUE roles : {[p.role_name for p in list_profiles("blue")]}')
print('Setup verified')

In [ ]:
# Cell 4 — Generate commander training dataset from real CIPHER v2 stub episodes.
#
# Strategy: run 200 stub-mode episodes with `CIPHER_AGENT_ARCH=v2`; on every step
# we capture (observation, commander_context) for both RED and BLUE commanders and
# emit one prompt per commander per step. The prompt mirrors what the live
# commander LLM sees in production.
import sys, os, json, copy
from datasets import Dataset

CIPHER_PATH = '/kaggle/working/CIPHER/cipher'
if CIPHER_PATH not in sys.path:
    sys.path.insert(0, CIPHER_PATH)
os.environ['LLM_MODE'] = 'stub'
os.environ['CIPHER_AGENT_ARCH'] = 'v2'

from cipher.utils.config import config
from cipher.environment.scenario import ScenarioGenerator
from cipher.training._episode_runner import run_episode
from cipher.agents.commander import RedCommander, BlueCommander
from cipher.agents.role_profiles import list_profiles

RED_COMMANDER_PROMPT = (
    open(os.path.join(CIPHER_PATH, 'cipher', 'agents', 'prompts', 'red_commander.txt'),
         encoding='utf-8').read()
)
BLUE_COMMANDER_PROMPT = (
    open(os.path.join(CIPHER_PATH, 'cipher', 'agents', 'prompts', 'blue_commander.txt'),
         encoding='utf-8').read()
)

RED_ROLES  = [p.role_name for p in list_profiles('red')]
BLUE_ROLES = [p.role_name for p in list_profiles('blue')]

META_ACTIONS = {'spawn_subagent', 'delegate_task', 'dismiss_subagent'}
VALID_RED_PRIM = {
    'move', 'read_file', 'exfiltrate', 'wait', 'abort',
    'plant_false_trail', 'plant_temporal_decoy', 'plant_honeypot_poison',
    'write_dead_drop', 'write_corrupted_drop', 'emergent',
}
VALID_BLUE_PRIM = {
    'investigate_node', 'trigger_alert', 'analyze_anomaly', 'reconstruct_path',
    'stand_down', 'place_honeypot', 'plant_breadcrumb', 'trigger_false_escalation',
    'emergent',
}

def _serialize_obs(obs) -> str:
    """Compact text snapshot of the observation; mirrors what _build_messages does live."""
    fields = {}
    for k in (
        'red_current_node', 'red_current_zone', 'red_suspicion_score',
        'blue_detection_confidence', 'red_path_length', 'context_resets_used',
        'current_detection_confidence', 'anomaly_feed', 'investigated_nodes',
        'recent_alerts', 'placed_honeypots',
    ):
        v = getattr(obs, k, None)
        if v is None:
            continue
        if isinstance(v, (list, tuple)):
            v = list(v)[:8]
        fields[k] = v
    return json.dumps(fields, default=str)[:2000]

def _commander_context(commander, roles_available):
    roster = commander.roster_snapshot()
    lines = '\n'.join(
        f"  - {r['id']} (role={r['role']}, lifespan={r['lifespan_remaining']})"
        for r in roster
    ) or '  (none — roster empty)'
    return (
        f"\n\nCOMMANDER CONTEXT ({commander.team.upper()}):\n"
        f"Available roles: {', '.join(roles_available)}.\n"
        f"Spawn budget remaining: {commander.registry.spawn_budget_remaining}. "
        f"Slots free: {commander.registry.max_concurrent - len(commander.registry)}.\n"
        f"Active roster ({len(roster)}):\n{lines}\n"
    )

def collect_commander_dataset(n_episodes=200, max_steps=20):
    """Run n_episodes of CIPHER v2 stub-mode and snapshot every commander step."""
    samples_red, samples_blue = [], []
    scen_gen = ScenarioGenerator()

    for ep in range(n_episodes):
        scenario = scen_gen.generate(ep + 1)
        # Construct fresh commanders so we control their lifecycle exactly.
        red_cmd = RedCommander('red_commander_01', config)
        blue_cmd = BlueCommander('blue_commander_01', config)
        red_cmd.reset_episode(); blue_cmd.reset_episode()

        # We use the runner in scripted-callback mode by patching commanders
        # with their pre-step observation hooks. Easier: just reuse run_episode
        # and pull commander state from the resulting trace.
        result = run_episode(
            scenario=scenario,
            graph=scenario.generated_graph,
            cfg=config,
            max_steps=max_steps,
            verbose=False,
            save_trace=False,
            episode_number=ep + 1,
        )
        state = result.get('state')
        if state is None:
            continue

        # Extract per-step prompts from episode_log + state snapshots.
        log = list(getattr(state, 'episode_log', []) or [])
        for entry in log:
            agent_id = str(entry.get('agent_id', ''))
            atype = str(entry.get('action_type', '') or '').lower()
            if not agent_id.startswith(('red_commander', 'blue_commander')):
                continue
            team = 'red' if agent_id.startswith('red') else 'blue'
            roles_available = RED_ROLES if team == 'red' else BLUE_ROLES

            # Synthetic observation snapshot — use whatever payload/fields exist.
            obs_text = json.dumps({
                'step': entry.get('step', 0),
                'red_zone': getattr(state, 'red_current_zone', 0),
                'red_suspicion': float(getattr(state, 'red_suspicion_score', 0.0)),
                'blue_detection': float(getattr(state, 'blue_detection_confidence', 0.0)),
                'exfil_count': len(getattr(state, 'red_exfiltrated_files', [])),
                'last_action': atype,
                'team': team,
            }, default=str)

            sys_prompt = (RED_COMMANDER_PROMPT if team == 'red' else BLUE_COMMANDER_PROMPT)
            sys_prompt = sys_prompt + (
                f"\n\nCOMMANDER CONTEXT ({team.upper()}):\n"
                f"Available roles: {', '.join(roles_available)}.\n"
                f"Spawn budget remaining: {entry.get('spawn_budget_left', 8)}.\n"
                f"Active roster size: {entry.get('roster_size', 0)}.\n"
            )

            conv = [
                {'role': 'system', 'content': sys_prompt},
                {'role': 'user',   'content': obs_text},
            ]
            try:
                prompt = tokenizer.apply_chat_template(
                    conv, tokenize=False, add_generation_prompt=True
                )
            except NameError:
                # Tokenizer not yet loaded — defer; we'll re-build inside the train cell.
                prompt = json.dumps(conv)

            sample = {
                'prompt':         prompt,
                'team':           team,
                'zone':           int(getattr(state, 'red_current_zone', 0)),
                'suspicion':      float(getattr(state, 'red_suspicion_score', 0.0)),
                'blue_detection': float(getattr(state, 'blue_detection_confidence', 0.0)),
                'roster_size':    int(entry.get('roster_size', 0) or 0),
                'spawn_budget':   int(entry.get('spawn_budget_left', 8) or 8),
                'last_action':    atype,
            }
            (samples_red if team == 'red' else samples_blue).append(sample)

        if (ep + 1) % 25 == 0:
            print(f'  ep {ep+1:3d}/{n_episodes}  red={len(samples_red):4d}  blue={len(samples_blue):4d}')

    print(f'Total RED samples : {len(samples_red)}')
    print(f'Total BLUE samples: {len(samples_blue)}')
    return samples_red, samples_blue

N_EPISODES = 200
MAX_STEPS  = 20
RED_SAMPLES, BLUE_SAMPLES = None, None  # filled inside the train cells once tokenizer exists
print(f'Dataset config ready: {N_EPISODES} episodes × {MAX_STEPS} steps per commander')
print('Datasets will be built once the tokenizer is loaded in cells 5 / 7.')

In [ ]:
# Cell 5 — Train RED Commander v1
import json, os, gc, torch
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

KAGGLE_OUT  = '/kaggle/working'
MAX_SEQ_LEN = 768

print('Loading unsloth/Llama-3.2-1B-Instruct')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded | Trainable params: {trainable:,}')

# Now we have a tokenizer — generate the datasets.
RED_SAMPLES, BLUE_SAMPLES = collect_commander_dataset(N_EPISODES, MAX_STEPS)
# Re-apply chat template now that tokenizer exists.
def _retemplate(samples, sys_prompt, roles_available):
    out = []
    for s in samples:
        sp = sys_prompt + (
            f"\n\nCOMMANDER CONTEXT ({s['team'].upper()}):\n"
            f"Available roles: {', '.join(roles_available)}.\n"
            f"Spawn budget remaining: {s['spawn_budget']}.\n"
            f"Active roster size: {s['roster_size']}.\n"
        )
        conv = [
            {'role': 'system', 'content': sp},
            {'role': 'user',
             'content': json.dumps({
                'zone': s['zone'], 'suspicion': s['suspicion'],
                'blue_detection': s['blue_detection'],
                'roster_size': s['roster_size'],
                'spawn_budget': s['spawn_budget'],
             })},
        ]
        s2 = dict(s)
        s2['prompt'] = tokenizer.apply_chat_template(
            conv, tokenize=False, add_generation_prompt=True
        )
        out.append(s2)
    return Dataset.from_list(out)

red_dataset = _retemplate(RED_SAMPLES, RED_COMMANDER_PROMPT, RED_ROLES)
print(f'RED commander dataset: {len(red_dataset)} prompts')

# ── Reward function for RED commander ────────────────────────
def red_commander_reward(prompts, completions, **kwargs):
    def _g(key, default, i):
        v = kwargs.get(key)
        if v is None: return default
        return v[i] if isinstance(v, (list, tuple)) and i < len(v) else default

    rewards = []
    for i, completion in enumerate(completions):
        zone        = int(_g('zone', 0, i))
        suspicion   = float(_g('suspicion', 0.3, i))
        blue_det    = float(_g('blue_detection', 0.0, i))
        roster_size = int(_g('roster_size', 0, i))
        spawn_left  = int(_g('spawn_budget', 8, i))

        txt = completion[-1]['content'] if isinstance(completion, list) else str(completion)
        txt = txt.strip()
        if txt.startswith('```'):
            txt = '\n'.join(l for l in txt.split('\n')
                             if not l.strip().startswith('```')).strip()
        try:
            data   = json.loads(txt)
            reward = 0.20
        except Exception:
            rewards.append(-0.40)
            continue

        action = str(data.get('action_type', '')).lower().strip()
        reasoning = str(data.get('reasoning', ''))
        if not action:
            rewards.append(-0.30); continue

        if action in META_ACTIONS or action in VALID_RED_PRIM:
            reward += 0.10
        else:
            rewards.append(-0.30); continue

        if len(reasoning) > 30: reward += 0.05
        if len(reasoning) > 80: reward += 0.05

        # ── Meta-action correctness ─────────────────────────────
        if action == 'spawn_subagent':
            spec = data.get('subagent_spec') or {}
            role = str(spec.get('role_name', '')).lower().strip()
            if role in RED_ROLES:
                reward += 0.30
            else:
                reward -= 0.20
            # Spawning when roster is empty / small is good early
            if roster_size < 3 and spawn_left > 0:
                reward += 0.15
            # Penalty for spawning past concurrency cap or with no budget
            if spawn_left <= 0 or roster_size >= 6:
                reward -= 0.40
            # Smart role selection by zone
            if role == 'exfiltrator' and zone < 2:
                reward -= 0.20
            if role == 'scout' and blue_det < 0.20 and roster_size < 6:
                reward += 0.10
            if role == 'planner' and zone <= 1:
                reward += 0.10

        elif action == 'delegate_task':
            tgt = str(data.get('target_subagent_id', ''))
            if tgt and roster_size > 0:
                reward += 0.20
            else:
                reward -= 0.20

        elif action == 'dismiss_subagent':
            tgt = str(data.get('target_subagent_id', ''))
            if tgt and roster_size > 0:
                reward += 0.15
            else:
                reward -= 0.15

        # ── Primitive zone-aware shaping (mirrors planner reward) ─
        elif action in VALID_RED_PRIM:
            if zone == 3 and action == 'exfiltrate' and data.get('target_file'):
                reward += 0.50
            elif zone in (1, 2) and action == 'move' and data.get('target_node') is not None:
                reward += 0.20
            elif action == 'wait' and suspicion < 0.40:
                reward -= 0.10
            elif action == 'abort' and not (suspicion > 0.80 and blue_det > 0.60):
                reward -= 0.30

        # Encourage commander to delegate when subagents already exist
        if roster_size >= 4 and action in VALID_RED_PRIM and action not in ('emergent',):
            reward -= 0.05  # delegation is preferred when roster is staffed

        rewards.append(float(max(-0.50, min(1.0, reward))))
    return rewards

red_save = f'{KAGGLE_OUT}/cipher-red-commander-v1'
cfg = GRPOConfig(
    output_dir=f'{KAGGLE_OUT}/cipher-red-commander-v1-ckpt',
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=192,
    max_prompt_length=512,
    temperature=0.7,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=10,
    save_steps=999999,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=42,
    warmup_ratio=0.05,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=cfg,
    train_dataset=red_dataset,
    reward_funcs=[red_commander_reward],
)
print('RED Commander v1 training started ...')
result = trainer.train()
print(f'Training complete | loss: {result.metrics.get("train_loss", "n/a")}')
model.save_pretrained(red_save)
tokenizer.save_pretrained(red_save)
print(f'Model saved: {red_save}')

In [ ]:
# Cell 6 — Clear GPU before BLUE Commander
import gc, torch

try:
    del model
    del trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

free_gb = torch.cuda.memory_reserved(0) / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
used_gb  = torch.cuda.memory_allocated(0) / 1e9
print(f'GPU memory cleared')
print(f'  Allocated : {used_gb:.2f} GB')
print(f'  Reserved  : {free_gb:.2f} GB')
print(f'  Total     : {total_gb:.2f} GB')
print(f'  Free est. : {total_gb - used_gb:.2f} GB')
print('Ready to load BLUE Commander base model')

In [ ]:
# Cell 7 — Train BLUE Commander v1
import json, os
from unsloth import FastLanguageModel
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

KAGGLE_OUT  = '/kaggle/working'
MAX_SEQ_LEN = 768

print('Loading unsloth/Llama-3.2-1B-Instruct (fresh)')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=7,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded | Trainable params: {trainable:,}')

blue_dataset = _retemplate(BLUE_SAMPLES, BLUE_COMMANDER_PROMPT, BLUE_ROLES)
print(f'BLUE commander dataset: {len(blue_dataset)} prompts')

def blue_commander_reward(prompts, completions, **kwargs):
    def _g(key, default, i):
        v = kwargs.get(key)
        if v is None: return default
        return v[i] if isinstance(v, (list, tuple)) and i < len(v) else default

    rewards = []
    for i, completion in enumerate(completions):
        zone        = int(_g('zone', 0, i))
        blue_det    = float(_g('blue_detection', 0.0, i))
        roster_size = int(_g('roster_size', 0, i))
        spawn_left  = int(_g('spawn_budget', 8, i))

        txt = completion[-1]['content'] if isinstance(completion, list) else str(completion)
        txt = txt.strip()
        if txt.startswith('```'):
            txt = '\n'.join(l for l in txt.split('\n')
                             if not l.strip().startswith('```')).strip()
        try:
            data   = json.loads(txt)
            reward = 0.20
        except Exception:
            rewards.append(-0.40); continue

        action = str(data.get('action_type', '')).lower().strip()
        reasoning = str(data.get('reasoning', ''))
        if not action:
            rewards.append(-0.30); continue

        if action in META_ACTIONS or action in VALID_BLUE_PRIM:
            reward += 0.10
        else:
            rewards.append(-0.30); continue

        if len(reasoning) > 30: reward += 0.05
        if len(reasoning) > 80: reward += 0.05

        if action == 'spawn_subagent':
            spec = data.get('subagent_spec') or {}
            role = str(spec.get('role_name', '')).lower().strip()
            if role in BLUE_ROLES:
                reward += 0.30
            else:
                reward -= 0.20
            if spawn_left <= 0 or roster_size >= 6:
                reward -= 0.40
            # Smart role choice based on detection confidence
            if role == 'alert_judge' and blue_det >= 0.55:
                reward += 0.20
            if role == 'surveillance' and roster_size == 0:
                reward += 0.20
            if role == 'forensics' and zone >= 2:
                reward += 0.10

        elif action == 'delegate_task':
            if data.get('target_subagent_id') and roster_size > 0:
                reward += 0.20
            else:
                reward -= 0.20

        elif action == 'dismiss_subagent':
            if data.get('target_subagent_id') and roster_size > 0:
                reward += 0.15
            else:
                reward -= 0.15

        elif action in VALID_BLUE_PRIM:
            if action == 'trigger_alert' and blue_det > 0.55:
                reward += 0.30
            elif action == 'trigger_alert' and blue_det < 0.30:
                reward -= 0.20
            elif action == 'investigate_node' and data.get('target_node') is not None:
                reward += 0.15
            elif action == 'place_honeypot' and zone <= 2:
                reward += 0.10

        if roster_size >= 4 and action in VALID_BLUE_PRIM and action != 'emergent':
            reward -= 0.05

        rewards.append(float(max(-0.50, min(1.0, reward))))
    return rewards

blue_save = f'{KAGGLE_OUT}/cipher-blue-commander-v1'
cfg = GRPOConfig(
    output_dir=f'{KAGGLE_OUT}/cipher-blue-commander-v1-ckpt',
    num_train_epochs=8,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=192,
    max_prompt_length=512,
    temperature=0.7,
    learning_rate=3e-5,
    fp16=True,
    logging_steps=10,
    save_steps=999999,
    report_to='none',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    seed=7,
    warmup_ratio=0.05,
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=cfg,
    train_dataset=blue_dataset,
    reward_funcs=[blue_commander_reward],
)
print('BLUE Commander v1 training started ...')
result = trainer.train()
print(f'Training complete | loss: {result.metrics.get("train_loss", "n/a")}')
model.save_pretrained(blue_save)
tokenizer.save_pretrained(blue_save)
print(f'Model saved: {blue_save}')

In [ ]:
# Cell 8 — Zip both LoRAs for download
import shutil, os

KAGGLE_OUT = '/kaggle/working'
for folder_name in ['cipher-red-commander-v1', 'cipher-blue-commander-v1']:
    folder_path = f'{KAGGLE_OUT}/{folder_name}'
    if os.path.exists(folder_path):
        shutil.make_archive(folder_path, 'zip', folder_path)
        size = os.path.getsize(f'{folder_path}.zip') / 1e6
        print(f'Zipped: {folder_name}.zip  ({size:.0f} MB)')
    else:
        print(f'MISSING: {folder_path} — check cells 5 and 7 ran successfully')

print()
print('All output files in /kaggle/working/:')
for f in sorted(os.listdir(KAGGLE_OUT)):
    fpath = os.path.join(KAGGLE_OUT, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {f}  ({size:.1f} MB)')

print()
print('=' * 60)
print('NEXT STEPS — drop these into your CIPHER project root:')
print('=' * 60)
print('  unzip cipher-red-commander-v1.zip -d "red trained/cipher-red-commander-v1"')
print('  unzip cipher-blue-commander-v1.zip -d "blue trained/cipher-blue-commander-v1"')
print()
print('Then `python main.py --hybrid` will pick them up via:')
print('  RED_COMMANDER_LORA_PATH=red trained/cipher-red-commander-v1')
print('  BLUE_COMMANDER_LORA_PATH=blue trained/cipher-blue-commander-v1')
print('Done.')